# Lab 2 - Người 3: Hyperparameter và fine-tuning 

Notebook này **không huấn luyện lại**. Nó lấy kết quả đã chạy trong nhánh `main` của `kakal1090/lab2-practice2-final` và phân tích phần của Người 3: (A) learning rate, (B) batch size, (C) fine-tuning `layer4` + `fc` so với baseline.

Điều kiện thực nghiệm của nhóm (theo notebook `PRACTICE_2_HOAN_CHINH`): CIFAR-10 đầy đủ, ảnh **96×96**, seed 42, 5 epoch mỗi cấu hình; kết quả có sẵn chạy bằng CPU. Các lượt head dùng feature cache của backbone cố định, lượt fine-tuning chạy trên ảnh.
Mọi con số trong báo cáo phải lấy từ các ô bên dưới (đọc trực tiếp từ CSV/JSON), không nhập tay. Không chọn tham số bằng test; test chỉ đọc từ `comparison.csv`.

## 1. Lấy kết quả từ GitHub
Ô dưới clone nhánh `main`, in commit và `environment.json` (ghi lại môi trường cho báo cáo). Nếu đã có repo trên máy thì đặt `LOCAL_PROJECT`.

In [ ]:
import os, sys, subprocess, json, csv, math, tempfile, statistics
from pathlib import Path

REPO = 'https://github.com/kakal1090/lab2-practice2-final.git'
LOCAL_PROJECT = None   # nếu đã có sẵn repo trên máy, đặt đường dẫn ở đây để bỏ qua bước clone
if LOCAL_PROJECT:
    project = Path(LOCAL_PROJECT)
else:
    base = Path('/content') if Path('/content').exists() else Path(tempfile.gettempdir())
    workspace = Path(tempfile.mkdtemp(prefix='p3_96_', dir=base))
    project = workspace / 'lab2-practice2-final'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO, str(project)], check=True)
RES = project / 'results'
assert (RES / 'comparison.csv').exists(), 'Thiếu results/comparison.csv'
head = subprocess.run(['git', '-C', str(project), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True)
print('Repo:', project)
print('Commit:', head.stdout.strip() or 'không xác định')
env_file = RES / 'environment.json'
print(env_file.read_text(encoding='utf-8') if env_file.exists() else 'Không có environment.json')

## 2. Liệt kê các run có trong `results/`
Xem tên thư mục từng run, các file có trong đó và `config.json`. Dùng bản in này để biết run nào là A (đổi learning rate), B (đổi batch size), C (fine-tuning) và giá trị thật của từng tham số; **không giả định lr 0.01 hay batch 64** nếu `config.json` ghi khác.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None

with (RES / 'comparison.csv').open(newline='', encoding='utf-8') as f:
    comparison = list(csv.DictReader(f))
print('comparison.csv - các cột:', list(comparison[0].keys()))
for row in comparison:
    print(row)

run_dirs = sorted(p.name for p in RES.iterdir() if p.is_dir())
print('\nThư mục run trong results/:', run_dirs)
for name in run_dirs:
    print(f'\n[{name}] file:', sorted(x.name for x in (RES / name).iterdir()))
    print('  config.json:', read_json(RES / name / 'config.json'))

## 3. Chọn run của Người 3
Tên `resnet_baseline` và `resnet_finetune` lấy từ notebook của nhóm. Hai run còn lại (đổi learning rate, đổi batch size) hãy điền đúng tên thư mục theo bản in ở mục 2.

In [ ]:
BASELINE = 'resnet_baseline'
RUN_FT = 'resnet_finetune'
RUN_LR = None      # điền tên thư mục run ResNet head đổi learning rate, ví dụ 'resnet_lr...'
RUN_BATCH = None   # điền tên thư mục run ResNet head đổi batch size

assert RUN_LR and RUN_BATCH, 'Điền RUN_LR và RUN_BATCH theo danh sách run ở mục 2'
P3_RUNS = [RUN_LR, RUN_BATCH, RUN_FT]
ALL = [BASELINE] + P3_RUNS
for r in ALL:
    assert (RES / r).is_dir(), f'Không có results/{r}'
print('Các run dùng trong phần 3:', ALL)

## 4. Kiểm tra điều kiện và lớp được mở
(1) `config.json` của mỗi run chỉ được khác baseline đúng yếu tố cần đổi (ngoài tên run/đường dẫn/thời gian); (2) các run phải cùng ảnh 96×96, seed 42, 5 epoch; (3) `parameters.json` cho biết số tham số và tên lớp trainable. Với ResNet18 + `Linear(512, 10)` số trainable kỳ vọng là 5.130 (head) và 8.393.728 + 5.130 = 8.398.858 (`layer4` + `fc`); các số này không phụ thuộc độ phân giải ảnh.

In [ ]:
def flat(d, prefix=''):
    out = {}
    for k, v in (d or {}).items():
        if isinstance(v, dict):
            out.update(flat(v, f'{prefix}{k}.'))
        else:
            out[f'{prefix}{k}'] = v
    return out

cfgs = {r: flat(read_json(RES / r / 'config.json')) for r in ALL}
for r in P3_RUNS:
    diff = {k: (cfgs[BASELINE].get(k), cfgs[r].get(k)) for k in sorted(set(cfgs[BASELINE]) | set(cfgs[r]))
            if cfgs[BASELINE].get(k) != cfgs[r].get(k)}
    print(f'{r}: khác baseline ở {diff}')

keys = ('seed', 'image', 'size', 'epoch', 'lr', 'learning', 'batch', 'mode', 'model')
print()
for r in ALL:
    print(r, {k: v for k, v in cfgs[r].items() if any(w in k.lower() for w in keys)})

EXPECTED = {'head': 5130, 'finetune': 5130 + 8393728}
def strip(n):
    return n.split('.', 1)[1] if n.startswith(('model.', 'module.')) else n
print()
for r in ALL:
    par = read_json(RES / r / 'parameters.json')
    names = [strip(n) for n in par['trainable_names']]
    is_ft = any(n.startswith('layer4.') for n in names)
    prefix_ok = all(n.startswith(('layer4.', 'fc.')) for n in names) if is_ft else all(n.startswith('fc.') for n in names)
    exp = EXPECTED['finetune' if is_ft else 'head']
    print(f"{r}: total={par['total']:,} | trainable={par['trainable']:,} ({100 * par['trainable'] / par['total']:.4f}%) | "
          f"{'finetune' if is_ft else 'head'} | tên trainable hợp lệ: {prefix_ok} | kỳ vọng {exp:,}: "
          f"{'KHỚP' if par['trainable'] == exp else 'KHÁC (xem architecture.txt)'}")

## 5. Bảng so sánh baseline + A + B + C
Bảng 1 là bảng bắt buộc (best validation accuracy, best epoch, train/validation loss tại best epoch, thời gian, trainable params, thêm test từ pipeline). Bảng 2 là chỉ số về dao động và overfitting. Best validation accuracy được tính lại từ lịch sử huấn luyện và **đối chiếu với `Val acc (%)` trong `comparison.csv`**; nếu lệch, mở file kiểm tra trước khi dùng số. Thời gian và test lấy từ `comparison.csv`.

Lưu ý khi so thời gian: các lượt head dùng feature cache của backbone còn lượt fine-tuning chạy trên ảnh, nên thời gian giữa head và fine-tuning không so sánh trực tiếp được; hãy kiểm tra trong `src/experiment.py` thời gian có gồm bước tạo cache không và nêu rõ trong báo cáo.

In [ ]:
def norm(s):
    return s.strip().lower().replace(' ', '_').replace('-', '_')

def pick(header, *tokens, avoid=()):
    for h in header:
        n = norm(h)
        if all(t in n for t in tokens) and not any(a in n for a in avoid):
            return h
    return None

def load_history(run):
    folder = RES / run
    path = folder / 'history.csv'
    if path.exists():
        with path.open(newline='', encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
        header = list(rows[0].keys())
        cols = {'tl': pick(header, 'train', 'loss'), 'ta': pick(header, 'train', 'acc'),
                'vl': pick(header, 'val', 'loss'), 'va': pick(header, 'val', 'acc')}
        missing = [k for k, v in cols.items() if v is None]
        if missing:
            raise KeyError(f'{run}: history.csv thiếu cột {missing}; các cột có: {header}')
        series = {k: [float(r[c]) for r in rows] for k, c in cols.items()}
        source = 'history.csv'
    else:
        events = sorted(folder.rglob('events.out.tfevents*'))
        if not events:
            raise FileNotFoundError(f'{run}: không có history.csv hay event TensorBoard. File có: {sorted(x.name for x in folder.iterdir())}')
        try:
            from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'tensorboard'], check=True)
            from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
        ea = EventAccumulator(str(events[0].parent), size_guidance={'scalars': 0})
        ea.Reload()
        tags = ea.Tags()['scalars']
        cols = {'tl': pick(tags, 'train', 'loss'), 'ta': pick(tags, 'train', 'acc'),
                'vl': pick(tags, 'val', 'loss'), 'va': pick(tags, 'val', 'acc')}
        missing = [k for k, v in cols.items() if v is None]
        if missing:
            raise KeyError(f'{run}: event log thiếu tag {missing}; các tag có: {tags}')
        series = {k: [e.value for e in ea.Scalars(t)] for k, t in cols.items()}
        source = 'TensorBoard events'
    scale = 100 if max(series['ta'] + series['va']) <= 1.0 else 1   # accuracy trong file có thể nằm trong [0, 1]
    return {'epoch': list(range(1, len(series['va']) + 1)), 'tl': series['tl'], 'vl': series['vl'],
            'ta': [x * scale for x in series['ta']], 'va': [x * scale for x in series['va']],
            'source': source, 'scale': scale}

def find_col(row, *tokens):
    for h in row:
        n = norm(h)
        if any(t in n for t in tokens):
            return h
    return None

official = {row['Run']: row for row in comparison}
time_col = find_col(comparison[0], 'time', 'thời_gian', 'sec', 'giây')
data, trainable = {}, {}
for r in ALL:
    data[r] = load_history(r)
    print(f"{r}: lấy từ {data[r]['source']}, {len(data[r]['va'])} epoch, nhân accuracy x{data[r]['scale']}")

main_rows, diag_rows = [], []
base_best = None
for r in ALL:
    h = data[r]
    i = max(range(len(h['va'])), key=lambda k: h['va'][k])   # epoch tốt nhất theo validation accuracy
    par = read_json(RES / r / 'parameters.json')
    trainable[r] = par['trainable']
    off = official.get(r, {})
    if r == BASELINE:
        base_best = h['va'][i]
    off_val = off.get('Val acc (%)')
    check = ''
    if off_val not in (None, ''):
        check = ' (khớp comparison.csv)' if abs(float(off_val) - h['va'][i]) < 0.011 else f' (LỆCH comparison.csv: {off_val})'
    main_rows.append([r, f"{h['va'][i]:.2f}{check}", '-' if r == BASELINE else f"{h['va'][i] - base_best:+.2f}",
                      f"{i + 1} (csv: {off.get('Best epoch', 'n/a')})", f"{h['tl'][i]:.4f}", f"{h['vl'][i]:.4f}",
                      off.get(time_col, 'n/a') if time_col else 'n/a', f"{par['trainable']:,}",
                      off.get('Test acc (%)') or 'không đánh giá'])
    diffs = [b - a for a, b in zip(h['va'], h['va'][1:])]
    diag_rows.append([r, f"{h['ta'][i]:.2f}", f"{h['ta'][i] - h['va'][i]:.2f}", f"{h['vl'][-1] - min(h['vl']):+.4f}",
                      f"{max([-d for d in diffs if d < 0], default=0.0):.2f}",
                      f"{statistics.pstdev(diffs) if diffs else 0.0:.2f}",
                      f"{sum(1 for a, b in zip(h['vl'], h['vl'][1:]) if b > a)}/{len(h['vl']) - 1}"])

def show(header, rows):
    print('| ' + ' | '.join(header) + ' |')
    print('|' + '---|' * len(header))
    for row in rows:
        print('| ' + ' | '.join(str(c) for c in row) + ' |')

print('\nBảng 1 - kết quả chính (accuracy %, chênh lệch tính bằng điểm phần trăm - pp)')
show(['run', 'best val acc (%)', 'Δ vs baseline (pp)', 'best epoch', 'train loss@best', 'val loss@best',
      'thời gian (comparison.csv)', 'trainable params', 'test acc (%)'], main_rows)
print('\nBảng 2 - chỉ số dao động / overfitting')
show(['run', 'train acc@best (%)', 'gap train-val (pp)', 'val loss cuối - min', 'val acc giảm lớn nhất (pp)',
      'độ lệch chuẩn Δval acc (pp)', 'số lần val loss tăng'], diag_rows)
print('\nLưu ý: một seed; chênh lệch nhỏ (dưới khoảng 1 pp) không đủ để kết luận chắc chắn.')

In [ ]:
import matplotlib.pyplot as plt
# Hình bổ sung vẽ từ lịch sử huấn luyện, cùng trục epoch. KHÔNG thay thế ảnh chụp TensorBoard (mục 6).
out_dir = Path('figures_p3'); out_dir.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (key, title) in zip(axes.flat, [('tl', 'train loss'), ('vl', 'validation loss'),
                                        ('ta', 'train accuracy (%)'), ('va', 'validation accuracy (%)')]):
    for r in ALL:
        ax.plot(data[r]['epoch'], data[r][key], marker='o', label=r, linestyle='--' if r == BASELINE else '-')
    ax.set_title(title); ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(out_dir / 'curves_4scalars_matplotlib.png', dpi=150)
plt.show()
print('Đã lưu:', out_dir / 'curves_4scalars_matplotlib.png')

## 6. TensorBoard thật
Event logs của từng run nằm trong `results/`. Đặt smoothing 0, xem đủ 5 epoch và chụp đủ 4 đường: `train_loss`, `val_loss`, `train_accuracy`, `val_accuracy`, giữ tên run và trục epoch. Lưu ảnh với tên rõ ràng để đưa vào báo cáo.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'tensorboard'], check=True)
get_ipython().run_line_magic('load_ext', 'tensorboard')
get_ipython().run_line_magic('tensorboard', f'--logdir "{RES}"')

## 7. Cấu hình thắng và test (đọc từ pipeline, không chạy thêm test)
Theo notebook nhóm, pipeline khóa lựa chọn bằng validation rồi mới test baseline và cấu hình thắng. Ô dưới xếp hạng theo validation và liệt kê run đã có test. Người 3 không chạy test riêng và không dùng test để chọn. Đặt `N_VAL` đúng với kích thước tập validation trong `split.json`/`config.json` (nếu vẫn là 5.000 ảnh thì giữ nguyên).

In [ ]:
N_VAL = 5000   # sửa theo tập validation thật của repo
ranked = sorted(comparison, key=lambda row: -float(row['Val acc (%)']))
print('Xếp hạng theo validation (comparison.csv):')
for k, row in enumerate(ranked, 1):
    print(f"{k}. {row['Run']}: val {row['Val acc (%)']}% | test {row['Test acc (%)'] or 'không đánh giá'}")
print('Run đã có test:', [row['Run'] for row in comparison if row['Test acc (%)']])
if len(ranked) > 1:
    p = float(ranked[0]['Val acc (%)']) / 100
    gap = float(ranked[0]['Val acc (%)']) - float(ranked[1]['Val acc (%)'])
    print(f"Chênh hạng 1 - hạng 2: {gap:.2f} pp | sai số chuẩn xấp xỉ của val accuracy với N_VAL={N_VAL}: {100 * math.sqrt(p * (1 - p) / N_VAL):.2f} pp")

## 8. Nhận xét của người thực hiện

> Điền sau khi chạy xong; xóa các dòng hướng dẫn (bắt đầu bằng `>`) và mọi chỗ `[...]` trước khi nộp. Số liệu lấy từ Bảng 1–2 (mục 5). Accuracy tính bằng %, chênh lệch tính bằng điểm phần trăm (pp). Gợi ý 400–600 từ.

**Điều kiện:** CIFAR-10, ảnh 96×96, seed 42, 5 epoch, kết quả chạy bằng [CPU/thiết bị trong `environment.json`], commit [...]. Lượt head dùng feature cache của backbone cố định; lượt fine-tuning chạy trên ảnh.

**A. Learning rate [giá trị baseline] → [giá trị mới] (`[RUN_LR]`).** Best val acc [..]% so với baseline [..]% (chênh [±..] pp), best epoch [..].
> Val loss/accuracy có dao động không? Nêu epoch có val loss tăng và mức giảm val accuracy lớn nhất ([..] pp) so với baseline.

**B. Batch size [..] → [..] (`[RUN_BATCH]`).** Best val acc [..]% (chênh [±..] pp).
> Số optimizer steps mỗi epoch = ceil(số ảnh train / batch size): [..] → [..], trong khi LR giữ nguyên, nên khác biệt có thể do ít bước cập nhật chứ không chỉ do batch size. Với lượt head dùng feature cache, thời gian của batch lớn chỉ phản ánh phần classifier; nêu rõ điều này khi bàn về thời gian.

**C. Fine-tuning `layer4` + `fc` (`[RUN_FT]`).** Trainable [..] tham số ([..]% tổng), best val acc [..]% (chênh [±..] pp), best epoch [..].
> Fine-tuning có cải thiện trong ngân sách 5 epoch không? Với ảnh 96×96, đầu ra của `layer4` có kích thước 3×3 (so với 7×7 ở 224×224), nên bàn về ảnh hưởng của độ phân giải chỉ như một giả thuyết, trừ khi có số liệu 224×224 cùng điều kiện (không có trong bảng này).

**Overfitting?** Gap train − val tại best epoch: baseline [..], A [..], B [..], C [..] pp; val loss cuối so với mức thấp nhất: [..].
> Chỉ kết luận có overfitting khi có bằng chứng trong số liệu (gap tăng, val loss tăng trong khi train loss giảm).

**Cấu hình thắng theo validation (pipeline):** [run], val [..]%; test: baseline [..]%, cấu hình thắng [..]% (đọc từ `comparison.csv`, không dùng test để chọn).

**Hạn chế:** một seed; validation [N_VAL] ảnh (1 pp ≈ [N_VAL/100] ảnh, sai số chuẩn khoảng 0,3–0,5 pp); chỉ 5 epoch; kết quả ở 96×96 không so trực tiếp với kết quả 224×224 cũ; thời gian giữa head (feature cache) và fine-tuning (chạy trên ảnh) không so sánh trực tiếp; [bổ sung].